# Data Preparation Notebook

**Project**: Intelligent Employee Task Allocation — DS-20 (Industry Explorer Track)  
**Client**: Cygnus One (Pvt) Ltd  

---

## Purpose of This Notebook

This notebook covers the initial **Data Collection** and **Data Understanding** phases of the project.

The goal is to:
1. Load the two raw datasets provided by Cygnus One
2. Inspect and understand their structure and contents
3. Merge them into a single combined dataset ready for EDA and preprocessing
4. Save the combined dataset for use in subsequent notebooks

---

## Source Datasets

| Dataset | File | Rows | Columns | Description |
|---|---|---|---|---|
| Timesheets | `Employee_Timesheets_Detailed_Dataset.csv` | 11,633 | 11 | One row per timesheet log entry — records who worked on what task, when, and for how many hours |
| Task Master | `Employee_Task_Allocation_Dataset.csv` | 3,356 | 18 | One row per unique task — contains task descriptions, priority, planned hours, stage, and assignment info |

---

## Step 1 — Import Libraries

We import `pandas` for all data loading, inspection, and transformation operations.

In [3]:
import pandas as pd

---

## Step 2 — Load Raw Datasets

We load both raw CSV files from the `data/raw/` directory and verify the row counts to confirm the files loaded correctly.

- **Timesheet dataset** (`Employee_Timesheets_Detailed_Dataset.csv`) — contains individual timesheet log entries.  
- **Task Master dataset** (`Employee_Task_Allocation_Dataset.csv`) — contains one record per unique task.

> These files are excluded from version control via `.gitignore` because they contain organisational data.

In [4]:
timesheet_path = '../data/raw/Employee_Timesheets_Detailed_Dataset.csv'
task_path = '../data/raw/Employee_Task_Allocation_Dataset.csv'
df_timesheet = pd.read_csv(timesheet_path)
df_tasks = pd.read_csv(task_path)
print(f"Timesheet Dataset Total Rows: {df_timesheet.shape[0]}")
print(f"Task Master Dataset Total Rows: {df_tasks.shape[0]}\n")

Timesheet Dataset Total Rows: 11633
Task Master Dataset Total Rows: 3356



---

## Step 3 — Preview & Understand Dataset Columns

We preview the first 5 rows of each dataset to understand the structure and content of every column.

### Timesheet Dataset — Column Descriptions (11 columns)

| Column | Type | Description | Plan |
|---|---|---|---|
| `Timesheet_ID` | String | Unique ID for each timesheet entry (e.g. `TS-12331`) | Reference key — drop before modelling |
| `Date` | Date | Date the work was logged | Extract temporal features (month, day-of-week) |
| `Task_ID` | String | Foreign key linking this log to a task (e.g. `TSK-1640`) | Join key with task master dataset |
| `Task_Name` | String | Short name/title of the task | Standardise & use as a categorical / NLP feature |
| `Work_Description` | String | Free-text description of the actual work done | Text preprocessing + TF-IDF feature |
| `Hours_Spent` | Float | Hours logged for this specific entry | Workload & experience aggregate feature |
| `Project_Name` | String | Project the task belongs to | Label-encode as categorical feature |
| `Employee_ID` | String | Unique employee identifier (e.g. `EMP-11`) | **Target variable** — who was assigned the task |
| `Employee_Name` | String | Full name of the employee | Reference only — drop before modelling |
| `Employee_Department` | String | Department of the employee (e.g. `Research and Development (R&D)`) | Label/one-hot encode as categorical feature |
| `Employee_Job_Position` | String | Job position/title of the employee | Label/one-hot encode as categorical feature |

### Task Master Dataset — Column Descriptions (18 columns)

| Column | Type | Description | Plan |
|---|---|---|---|
| `Task_ID` | String | Unique task identifier (e.g. `TSK-1`) | Join key — kept after merge |
| `Task_Title` | String | Short title (may overlap with `Task_Name` in timesheet) | **Dropped** — redundant with `Task_Name` |
| `Project_Name` | String | Project the task belongs to | **Dropped** — already exists in timesheet |
| `Task_Description` | String | Full description of the task (includes work details) | **Primary NLP feature** — TF-IDF |
| `Timesheet_Work_Logs` | String | Aggregated pipe-separated work log entries for the task | Secondary NLP feature — combine with `Task_Description` |
| `Original_Task_Description` | String | Original unprocessed task description (many nulls) | May be dropped after EDA — high null rate |
| `Task_Priority` | String | Priority level (`Low`, `Normal`, `High`, `Urgent`) | Ordinal encode (Low=0 … Urgent=3) |
| `Estimated_Planned_Hours` | Float | Planned hours for the task | Numerical feature — normalise |
| `Actual_Hours_Spent` | Float | Total hours actually spent on the task | **Leakage risk** — investigate carefully |
| `Timesheet_Logs_Count` | Int | Number of timesheet entries for this task | Task complexity proxy — numerical feature |
| `Task_Stage` | String | Current stage (`To Do`, `Ongoing`, `Done`, etc.) | **Leakage risk** — only use if it reflects assignment stage, not completion |
| `Created_Date` | Date | Date the task was created | Extract temporal features; compute task age |
| `Deadline_Date` | Date | Deadline for the task (many nulls) | Derive `has_deadline` binary flag; compute urgency |
| `Assigned_Employee_ID` | String | Employee ID assigned to the task | **Dropped** — duplicates `Employee_ID` from timesheet |
| `Assigned_Employee_Name` | String | Name of the assigned employee | **Dropped** — duplicates `Employee_Name` |
| `Employee_Department` | String | Department of the assigned employee | **Dropped** — already exists in timesheet |
| `Employee_Job_Position` | String | Job position of the assigned employee | **Dropped** — already exists in timesheet |
| `All_Collaborating_Employees` | String | All employees who logged time on this task | Derive `collaborator_count` feature |

In [5]:
print(f"--- Timesheet Dataset (First 5 Rows, {df_timesheet.shape[1]} Columns) ---")
display(df_timesheet.head())
print(f"\n--- Task Master Dataset (First 5 Rows, {df_tasks.shape[1]} Columns) ---")
display(df_tasks.head())

--- Timesheet Dataset (First 5 Rows, 11 Columns) ---


,Timesheet_ID,Date,Task_ID,Task_Name,Work_Description,Hours_Spent,Project_Name,Employee_ID,Employee_Name,Employee_Department,Employee_Job_Position
0,TS-12331,2026-09-14,TSK-1640,Odoo SH Maintain,db backup and restore in test,0.25,Hovael Project,EMP-11,W M I L Wijesinghe,Research and Development (R&D),Team Lead - Research and Development (R&D)
1,TS-12330,2026-09-14,TSK-1881,Odoo sh Maintain,db backup and restore in test,0.25,CEYLON ECO SPICES,EMP-11,W M I L Wijesinghe,Research and Development (R&D),Team Lead - Research and Development (R&D)
2,TS-12329,2026-09-14,TSK-182,Odoo.SH Maintaing,add addons,0.25,Mihiri Bakemart (Pvt)Ltd - Development,EMP-11,W M I L Wijesinghe,Research and Development (R&D),Team Lead - Research and Development (R&D)
3,TS-12328,2026-09-14,TSK-2884,Development Meeting,/,0.00,Cygnus One,EMP-55,Malshi Jayanthi,Colombo Branch,Project Manager
4,TS-12327,2026-09-14,TSK-405,Other Tasks (Mention on description),get privillages to charith's new github accoun...,0.50,Cygnus One,EMP-11,W M I L Wijesinghe,Research and Development (R&D),Team Lead - Research and Development (R&D)



--- Task Master Dataset (First 5 Rows, 18 Columns) ---


,Task_ID,Task_Title,Project_Name,Task_Description,Timesheet_Work_Logs,Original_Task_Description,Task_Priority,Estimated_Planned_Hours,Actual_Hours_Spent,Timesheet_Logs_Count,Task_Stage,Created_Date,Deadline_Date,Assigned_Employee_ID,Assigned_Employee_Name,Employee_Department,Employee_Job_Position,All_Collaborating_Employees
0,TSK-1,Training,Internal,Training. Work Details: Analysis | Inventory M...,Analysis | Inventory Module 'How to work locat...,NaN,Low,0.0,23.0,8,Internal,2025-04-16,NaN,EMP-18,E I S Dhananjaya,Business Solution,Assistent Project Manager,"H M N S K Herath, Administrator, D.R.perera, A..."
1,TSK-2,Meeting,Internal,Meeting. Work Details: Analysis | Mihira UAT |...,Analysis | Mihira UAT | at the main office Kur...,NaN,Low,0.0,47.0,25,Internal,2025-04-16,NaN,EMP-18,E I S Dhananjaya,Business Solution,Assistent Project Manager,"Administrator, A R M S Madusanka, R P L Shanth..."
2,TSK-3,Time Off,Internal,Time Off. Work Details: Time Off (1/2) | Time ...,Time Off (1/2) | Time Off (2/2) | Time Off (1/...,NaN,Low,0.0,642.0,84,Internal,2025-04-16,NaN,EMP-10,M H R Chandrasoma,Research and Development (R&D),Associate Software Engineer,"H.M.C.S Thilakarathna, L H P S S Pathirana, I ..."
3,TSK-16,Odoo 17 ERP Implementation,S00140 - Odoo 17 Implementation,Odoo 17 ERP Implementation in project S00140 -...,NaN,NaN,Low,0.0,0.0,0,To Do,2025-05-28,NaN,EMP-1,Administrator,Administration,Software Engineer,Administrator
4,TSK-17,Handover from Sales,Mihiri Bakemart (Pvt) Ltd Implementation,Handover from Sales in project Mihiri Bakemart...,NaN,NaN,Low,0.0,0.0,0,Project Preparation,2025-05-29,NaN,EMP-1,Administrator,Administration,Software Engineer,Administrator


---

## Step 4 — Merge Datasets

### Why We Merge

The two datasets complement each other:
- The **Timesheet dataset** tells us *who worked on what task and when*, but has limited task detail.
- The **Task Master dataset** contains full *task descriptions, priority, planned hours, and stage*, but lacks individual work log entries.

By joining on `Task_ID`, we create a single rich dataset that gives each timesheet entry the full context of its underlying task.

### Join Strategy

We perform a **Left Join** (timesheet ← task master) on `Task_ID` so that:
- Every timesheet entry is retained (even if the task has no match in the task master).
- Task-level details are enriched into each row.

### Columns Dropped Before Merging

To avoid duplicate columns in the combined dataset, the following columns are dropped from the **Task Master dataset** before the join:

| Dropped Column | Reason |
|---|---|
| `Task_Title` | Duplicates `Task_Name` from the timesheet |
| `Project_Name` | Already present in the timesheet dataset |
| `Assigned_Employee_ID` | Duplicates `Employee_ID` from the timesheet |
| `Assigned_Employee_Name` | Duplicates `Employee_Name` from the timesheet |
| `Employee_Department` | Already present in the timesheet dataset |
| `Employee_Job_Position` | Already present in the timesheet dataset |

> After dropping these 6 columns, the Task Master is reduced to **12 columns** before the join.

In [6]:
columns_to_drop = [
    'Task_Title',
    'Project_Name',
    'Assigned_Employee_ID',
    'Assigned_Employee_Name',
    'Employee_Department',
    'Employee_Job_Position'
]
df_tasks_clean = df_tasks.drop(columns=columns_to_drop)
df_combined = pd.merge(
    df_timesheet, 
    df_tasks_clean, 
    on='Task_ID', 
    how='left'
)

---

## Step 5 — Verify the Combined Dataset

After merging, we verify that:
- The row count is still **11,633** (left join preserves all timesheet rows).
- The column count is **22** (11 timesheet + 12 task − 1 shared `Task_ID` join key).

### Combined Dataset — Final Column List (22 columns)

| # | Column | Source | Next Steps |
|---|---|---|---|
| 1 | `Timesheet_ID` | Timesheet | Drop before modelling |
| 2 | `Date` | Timesheet | Extract month, day-of-week features |
| 3 | `Task_ID` | Both (join key) | Keep for grouping; drop before modelling |
| 4 | `Task_Name` | Timesheet | Standardise; encode as categorical |
| 5 | `Work_Description` | Timesheet | Clean text; TF-IDF feature |
| 6 | `Hours_Spent` | Timesheet | Normalise; aggregate per employee |
| 7 | `Project_Name` | Timesheet | Label-encode |
| 8 | `Employee_ID` | Timesheet | **Target variable** |
| 9 | `Employee_Name` | Timesheet | Drop before modelling |
| 10 | `Employee_Department` | Timesheet | Encode |
| 11 | `Employee_Job_Position` | Timesheet | Encode |
| 12 | `Task_Description` | Task Master | Primary NLP feature (TF-IDF) |
| 13 | `Timesheet_Work_Logs` | Task Master | Secondary NLP feature |
| 14 | `Original_Task_Description` | Task Master | Likely drop (high null rate) |
| 15 | `Task_Priority` | Task Master | Ordinal encode |
| 16 | `Estimated_Planned_Hours` | Task Master | Normalise |
| 17 | `Actual_Hours_Spent` | Task Master | Investigate leakage risk |
| 18 | `Timesheet_Logs_Count` | Task Master | Complexity proxy feature |
| 19 | `Task_Stage` | Task Master | Investigate leakage risk |
| 20 | `Created_Date` | Task Master | Extract temporal features |
| 21 | `Deadline_Date` | Task Master | Derive `has_deadline`; compute urgency |
| 22 | `All_Collaborating_Employees` | Task Master | Derive `collaborator_count` feature |

In [7]:
print(f"Combined Dataset Total Rows: {df_combined.shape[0]}")
print(f"Combined Dataset Total Columns: {df_combined.shape[1]}\n")
print("--- Combined Dataset (First 5 Rows) ---")
display(df_combined.head())

Combined Dataset Total Rows: 11633
Combined Dataset Total Columns: 22

--- Combined Dataset (First 5 Rows) ---


,Timesheet_ID,Date,Task_ID,Task_Name,Work_Description,Hours_Spent,Project_Name,Employee_ID,Employee_Name,Employee_Department,...,Timesheet_Work_Logs,Original_Task_Description,Task_Priority,Estimated_Planned_Hours,Actual_Hours_Spent,Timesheet_Logs_Count,Task_Stage,Created_Date,Deadline_Date,All_Collaborating_Employees
0,TS-12331,2026-09-14,TSK-1640,Odoo SH Maintain,db backup and restore in test,0.25,Hovael Project,EMP-11,W M I L Wijesinghe,Research and Development (R&D),...,Add addons t the server | Get backup from live...,NaN,Low,0.0,14.25,27,Project Preparation,2026-02-05,NaN,W M I L Wijesinghe
1,TS-12330,2026-09-14,TSK-1881,Odoo sh Maintain,db backup and restore in test,0.25,CEYLON ECO SPICES,EMP-11,W M I L Wijesinghe,Research and Development (R&D),...,add addon and test | get odoo sh live ackup an...,NaN,Low,0.0,6.50,13,Developments,2026-03-19,NaN,W M I L Wijesinghe
2,TS-12329,2026-09-14,TSK-182,Odoo.SH Maintaing,add addons,0.25,Mihiri Bakemart (Pvt)Ltd - Development,EMP-11,W M I L Wijesinghe,Research and Development (R&D),...,add all addon and build and test on the odoo s...,NaN,Low,0.0,40.75,29,Ongoing,2025-06-09,NaN,W M I L Wijesinghe
3,TS-12328,2026-09-14,TSK-2884,Development Meeting,/,0.00,Cygnus One,EMP-55,Malshi Jayanthi,Colombo Branch,...,intern development hoveal project meeting | me...,NaN,Low,0.0,9.42,13,Miscellaneous,2026-07-06,NaN,"W M I L Wijesinghe, K R V Dias, A R M S Madusa..."
4,TS-12327,2026-09-14,TSK-405,Other Tasks (Mention on description),get privillages to charith's new github accoun...,0.50,Cygnus One,EMP-11,W M I L Wijesinghe,Research and Development (R&D),...,self ssl setup on vps | Preparing Report List ...,NaN,Low,0.0,162.18,69,Miscellaneous,2025-07-14,NaN,"H.M.C.S Thilakarathna, Sadaruwan Bandara, L H ..."


---

## Step 6 — Save Combined Dataset

We save the combined dataset back to `data/raw/` as `Combined_Employee_Task_Data.csv`.

> This file will be the **primary input** for the next notebook (EDA & Preprocessing).

> Note: The entire `data/` directory is excluded from version control via `.gitignore`.

In [8]:
output_path = '../data/raw/Combined_Employee_Task_Data.csv'
df_combined.to_csv(output_path, index=False)
print(f"Combined dataset successfully saved to: {output_path}")

Combined dataset successfully saved to: ../data/raw/Combined_Employee_Task_Data.csv


---

## What Has Been Done in This Notebook

The following tasks from the project plan have been **completed** in this notebook:

### Data Collection
- [x] Raw datasets obtained from Cygnus One and stored in `data/raw/`
- [x] `Employee_Timesheets_Detailed_Dataset.csv` loaded — **11,633 rows, 11 columns**
- [x] `Employee_Task_Allocation_Dataset.csv` loaded — **3,356 rows, 18 columns**

### Initial Data Understanding
- [x] Row counts verified for both datasets
- [x] Column structure inspected (names, data types, sample values)
- [x] All 29 columns across both datasets catalogued with descriptions and planned usage

### Data Merging
- [x] 6 redundant/duplicate columns identified and dropped from the Task Master dataset before merging
- [x] Left join performed on `Task_ID` (timesheet ← task master)
- [x] Combined dataset produced: **11,633 rows, 22 columns**
- [x] Combined dataset verified (shape confirmed, first 5 rows reviewed)
- [x] Combined dataset saved to `data/raw/Combined_Employee_Task_Data.csv`

---